# Logistic regression baseline
Logistic regression predicts the probability that a transaction is illicit (`label = 1`). It provides a simple linear baseline for comparison with the Random Forest.

This notebook uses the same chronological splits from `validation_splits.json`. Each split fits a **new** scaler and classifier using only that split's labelled training transactions. Unknown labels are excluded from supervised fitting and scoring. `txId` and `time_step` are used for checks but are not predictors.

The final section trains on all known labels from steps 1–35 and creates a Kaggle submission for steps 36–49. Required packages: pandas, numpy, scikit-learn, and a Python Jupyter kernel.

## Findings: Not proceeding with this model -> mean AUC-ROC only 0.90

In [1]:
from pathlib import Path
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "bt-4012-competition-2026").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the competition project.")
DATA = ROOT / "bt-4012-competition-2026"
SPLITS = json.loads((ROOT / "validation_splits.json").read_text())
FEATURES = [f"feat_{i}" for i in range(1, 166)]
MODEL_PARAMS = dict(C=1.0, class_weight=None, max_iter=2000, random_state=42)
THRESHOLD = 0.5

def new_model():
    # The scaler is fitted independently within each fold through this pipeline.
    return make_pipeline(StandardScaler(), LogisticRegression(**MODEL_PARAMS))

print("Python:", platform.python_version(), "scikit-learn:", sklearn.__version__)
display(pd.DataFrame(SPLITS))

Python: 3.13.1 scikit-learn: 1.7.1


,name,train_start,train_end,validation_start,validation_end
0,earlier check,1,21,22,28
1,latest holdout,1,28,29,35


## Load and check the training data
Features are numeric and pre-normalized, but a training-fold StandardScaler makes the logistic regression's regularization more consistent across features. No values are imputed. The chosen settings (`C=1.0`, no class weights, threshold 0.5) define the initial baseline; any later adjustment should be a separate experiment.

In [2]:
train = pd.read_csv(DATA / "train.csv", usecols=["txId", "time_step", *FEATURES, "label"])
unexpected = train.loc[train.label.notna() & ~train.label.isin([0, 1]), "label"].unique()
if len(unexpected):
    raise ValueError(f"Unexpected known labels: {unexpected}")
known = train.loc[train.label.isin([0, 1])].copy()
known["label"] = known["label"].astype(int)
assert train.txId.is_unique, "Duplicate train transaction IDs need investigation."
assert np.isfinite(train[FEATURES].to_numpy()).all(), "Training features contain missing/infinite values."
print(f"All training transactions: {len(train):,}")
print(f"Known labels used for supervised learning: {len(known):,}")
print(f"Unknown labels excluded: {train.label.isna().sum():,}")
display(known.label.value_counts().rename_axis("label").to_frame("transactions"))

All training transactions: 141,772
Known labels used for supervised learning: 31,235
Unknown labels excluded: 110,537


,transactions
label,
0,27591
1,3644


## Evaluate on both chronological splits
ROC-AUC and average precision use predicted probabilities. Precision, recall, F1, and confusion matrices use the stated, untuned 0.5 threshold. Each fold starts with a new scaler and a new logistic regression model; validation labels are used only for scoring.

In [3]:
results = []
confusions = {}
for split in SPLITS:
    training = known.loc[known.time_step.between(split["train_start"], split["train_end"])]
    validation = known.loc[known.time_step.between(split["validation_start"], split["validation_end"])]
    assert split["train_end"] < split["validation_start"], "Overlapping periods."
    assert set(training.txId).isdisjoint(validation.txId), "Transaction overlap."
    assert training.label.nunique() == validation.label.nunique() == 2, "Both classes required."

    model = new_model()
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(training[FEATURES], training.label)
    fit_seconds = time.perf_counter() - started
    illicit_column = list(model[-1].classes_).index(1)
    p = model.predict_proba(validation[FEATURES])[:, illicit_column]
    predicted = (p >= THRESHOLD).astype(int)

    results.append({
        "split": split["name"],
        "train_steps": f'{split["train_start"]}–{split["train_end"]}',
        "validation_steps": f'{split["validation_start"]}–{split["validation_end"]}',
        "train_rows": len(training), "validation_rows": len(validation),
        "validation_illicit": int(validation.label.sum()),
        "validation_illicit_share": float(validation.label.mean()),
        "validation_roc_auc": roc_auc_score(validation.label, p),
        "average_precision": average_precision_score(validation.label, p),
        "threshold": THRESHOLD,
        "illicit_precision": precision_score(validation.label, predicted, zero_division=0),
        "illicit_recall": recall_score(validation.label, predicted, zero_division=0),
        "illicit_f1": f1_score(validation.label, predicted, zero_division=0),
        "fit_seconds": fit_seconds,
    })
    confusions[split["name"]] = pd.DataFrame(
        confusion_matrix(validation.label, predicted, labels=[0, 1]),
        index=["Actual licit", "Actual illicit"],
        columns=["Predicted licit", "Predicted illicit"],
    )
    print(f'{split["name"]}: ROC-AUC = {results[-1]["validation_roc_auc"]:.4f}; '
          f'fit time = {fit_seconds:.1f}s', flush=True)

results_df = pd.DataFrame(results)
with pd.option_context("display.max_columns", None):
    display(results_df.round(4))
for name, matrix in confusions.items():
    print(name)
    display(matrix)

earlier check: ROC-AUC = 0.9093; fit time = 6.9s


latest holdout: ROC-AUC = 0.9035; fit time = 10.9s


,split,train_steps,validation_steps,train_rows,validation_rows,validation_illicit,validation_illicit_share,validation_roc_auc,average_precision,threshold,illicit_precision,illicit_recall,illicit_f1,fit_seconds
0,earlier check,1–21,22–28,19530,5677,671,0.1182,0.9093,0.6223,0.5,0.5518,0.7541,0.6373,6.9388
1,latest holdout,1–28,29–35,25207,6028,1102,0.1828,0.9035,0.6313,0.5,0.5233,0.7350,0.6113,10.9129


earlier check


,Predicted licit,Predicted illicit
Actual licit,4595,411
Actual illicit,165,506


latest holdout


,Predicted licit,Predicted illicit
Actual licit,4188,738
Actual illicit,292,810


## Save the local validation results
Keep the two fold scores visible; their average alone can hide period-specific performance. The private Kaggle score is unknown, and the public score will be entered only after a submission is evaluated.

In [4]:
print(f'Mean validation ROC-AUC: {results_df.validation_roc_auc.mean():.4f}')
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
results_path = RESULTS / "logistic_regression_baseline.csv"
results_df.to_csv(results_path, index=False)
settings = {
    "model": "StandardScaler + LogisticRegression",
    "parameters": MODEL_PARAMS,
    "features": FEATURES,
    "threshold": THRESHOLD,
    "splits": SPLITS,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
}
(RESULTS / "logistic_regression_baseline_settings.json").write_text(
    json.dumps(settings, indent=2) + "\n"
)
print("Saved validation metrics and model settings in results/.")

Mean validation ROC-AUC: 0.9064
Saved validation metrics and model settings in results/.


## Fit on all known labels and create a Kaggle submission
After evaluation, train a fresh pipeline on all 31,235 known labels from steps 1–35. The test rows are from later steps. Save the probability of **class 1** for each test `index`, in the exact order of `sample_submission.csv`. The submission file is ready for you to upload to Kaggle.

In [5]:
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample.columns) == ["index", "target"], "Unexpected submission columns."
assert test["index"].is_unique and sample["index"].is_unique
assert len(test) == len(sample) and set(test["index"]) == set(sample["index"])
assert test.txId.is_unique and set(train.txId).isdisjoint(test.txId)
assert test.time_step.min() > train.time_step.max(), "Expected future test periods."
assert np.isfinite(test[FEATURES].to_numpy()).all(), "Test features contain missing/infinite values."

final_model = new_model()
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    final_model.fit(known[FEATURES], known.label)
illicit_column = list(final_model[-1].classes_).index(1)
probabilities = final_model.predict_proba(test[FEATURES])[:, illicit_column]
probability_by_index = pd.Series(probabilities, index=test["index"])
submission = sample.copy()
submission["target"] = submission["index"].map(probability_by_index)
assert np.isfinite(submission.target).all() and submission.target.between(0, 1).all()

SUBMISSIONS = ROOT / "submissions"
SUBMISSIONS.mkdir(exist_ok=True)
submission_path = SUBMISSIONS / "logistic_regression_baseline.csv"
submission.to_csv(submission_path, index=False)

saved = pd.read_csv(submission_path)
assert list(saved.columns) == ["index", "target"]
assert len(saved) == len(test) and saved["index"].equals(sample["index"])
assert np.isfinite(saved.target).all() and saved.target.between(0, 1).all()
assert np.allclose(saved.target, submission.target, rtol=0, atol=1e-12)
print(f"Final model trained on {len(known):,} known labels.")
print(f"Saved {len(saved):,} test probabilities to submissions/{submission_path.name}")
display(saved.head())
display(saved.target.describe())

Final model trained on 31,235 known labels.
Saved 15,329 test probabilities to submissions/logistic_regression_baseline.csv


,index,target
0,0,0.009326
1,1,0.020207
2,2,0.001469
3,3,0.004181
4,4,0.000263


count     1.532900e+04
mean      1.341136e-01
std       2.498932e-01
min      1.105982e-274
25%       1.311291e-09
50%       5.330706e-04
75%       1.339023e-01
max       1.000000e+00
Name: target, dtype: float64